# MV-Adapter · VS Code + Google Colab

这份 notebook 保存在本地 Git 仓库，但 cell 由 VS Code 的官方 Colab 内核在远端执行。代码每次会话克隆到 `/content`，模型和输出持久化到 Google Drive。默认运行显存压力较小的 SD2.1 文生多视图模型。

## 1. 挂载 Google Drive

In [12]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("当前 kernel 不是 Google Colab；请在右上角 Select Kernel > Colab 中连接。") from exc

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 配置 GitHub、Drive 和模型

`GITHUB_REPO` 指向你的 fork；若仓库或分支以后变更，只改这里。`HF_HOME` 放在 Drive，因此基础模型下载一次后可以跨 Colab 会话复用。

In [13]:
import os
from pathlib import Path

GITHUB_REPO = "https://github.com/WANG-Ruipeng/MV-Adapter.git"
BRANCH = "main"
COLAB_REPO = Path("/content/MV-Adapter")

MODEL_ROOT = Path("/content/drive/MyDrive/ModelWeights/MV-Adapter")
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/Colab_Projects/MV-Adapter")
ADAPTER_DIR = MODEL_ROOT / "mv-adapter"
HF_HOME = MODEL_ROOT / "huggingface"
OUTPUT_DIR = PROJECT_DRIVE_ROOT / "outputs"

BASE_MODEL = "stabilityai/stable-diffusion-2-1-base"
ADAPTER_REPO = "huanngzh/mv-adapter"
ADAPTER_WEIGHT = "mvadapter_t2mv_sd21.safetensors"

for path in (ADAPTER_DIR, HF_HOME, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")

print(f"临时代码: {COLAB_REPO}")
print(f"Adapter:  {ADAPTER_DIR}")
print(f"HF cache: {HF_HOME}")
print(f"输出目录: {OUTPUT_DIR}")

临时代码: /content/MV-Adapter
Adapter:  /content/drive/MyDrive/ModelWeights/MV-Adapter/mv-adapter
HF cache: /content/drive/MyDrive/ModelWeights/MV-Adapter/huggingface
输出目录: /content/drive/MyDrive/Colab_Projects/MV-Adapter/outputs


## 4. 检查 Colab GPU

从这里开始才需要 GPU；只下载 checkpoint 时不要运行此 cell。

In [14]:
import subprocess
import torch

assert torch.cuda.is_available(), "没有检测到 CUDA GPU；请换成带 GPU 的 Colab Server。"
subprocess.run(["nvidia-smi"], check=True)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU: NVIDIA L4


## 5. 在 Colab 临时磁盘克隆或更新代码

本地 workspace 不会自动变成 Colab 的文件系统，所以远端必须从 GitHub 获取代码。请先把本地需要运行的改动 push 到 `origin/main`。

In [15]:
import subprocess

if (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(COLAB_REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
elif COLAB_REPO.exists():
    raise RuntimeError(f"{COLAB_REPO} 已存在但不是 Git 仓库，请更换 COLAB_REPO 或手动清理。")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, GITHUB_REPO, str(COLAB_REPO)], check=True)

subprocess.run(["git", "-C", str(COLAB_REPO), "log", "-1", "--oneline"], check=True)

CompletedProcess(args=['git', '-C', '/content/MV-Adapter', 'log', '-1', '--oneline'], returncode=0)

## 6. 安装推理依赖

这里使用 `requirements-colab.txt`，跳过训练和纹理生成才需要的 `CV-CUDA`、Open3D、PyMeshLab 等组件。首次安装会花几分钟。

In [16]:
import sys
from pathlib import Path

def run_pip(*args):
    command = [sys.executable, "-m", "pip", *args]
    print("\n$", " ".join(map(str, command)), flush=True)
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout, flush=True)
    result.check_returncode()

requirements_file = COLAB_REPO / "requirements-colab.txt"
base_requirements_file = Path("/tmp/mvadapter-requirements-colab.txt")
base_requirements = [
    line for line in requirements_file.read_text(encoding="utf-8").splitlines()
    if not line.strip().startswith(("ninja", "nvdiffrast"))
]
base_requirements_file.write_text("\n".join(base_requirements) + "\n", encoding="utf-8")

run_pip("install", "-r", str(base_requirements_file))
run_pip("install", "setuptools", "wheel", "ninja")
run_pip("install", "git+https://github.com/NVlabs/nvdiffrast.git", "--no-build-isolation")
run_pip("install", "-e", str(COLAB_REPO), "--no-deps")
print("依赖和 MV-Adapter 已安装。")

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/MV-Adapter/requirements-colab.txt']' returned non-zero exit status 1.

## 7. 运行一次 SD2.1 文生多视图推理

基础模型会由 Diffusers 下载到上面配置的 Drive Hugging Face cache。第一次慢，后续会话会复用。

In [ ]:
from IPython.display import Image, display

PROMPT = "a small red robot, studio lighting, high quality"
OUTPUT_FILE = OUTPUT_DIR / "t2mv_sd21.png"
adapter_file = ADAPTER_DIR / ADAPTER_WEIGHT
assert adapter_file.is_file(), f"缺少 {adapter_file}；请先运行 checkpoint 下载 cell。"

command = [
    sys.executable,
    "-m",
    "scripts.inference_t2mv_sd",
    "--base_model",
    BASE_MODEL,
    "--adapter_path",
    str(ADAPTER_DIR),
    "--text",
    PROMPT,
    "--num_inference_steps",
    "30",
    "--seed",
    "42",
    "--output",
    str(OUTPUT_FILE),
]
subprocess.run(command, cwd=COLAB_REPO, env=os.environ.copy(), check=True)
print(f"结果已保存到: {OUTPUT_FILE}")
display(Image(filename=str(OUTPUT_FILE)))

## 切换到 SDXL

显存足够时，把 `BASE_MODEL` 改为 `stabilityai/stable-diffusion-xl-base-1.0`，把 `ADAPTER_WEIGHT` 改为 `mvadapter_t2mv_sdxl.safetensors`，并把推理模块改为 `scripts.inference_t2mv_sdxl`。重新运行配置、checkpoint 下载和推理 cell 即可。